# Gender Bias Detection - Token Classification Fine-tuning

This notebook fine-tunes a model for sentence-level gender bias detection using token classification (BIO tagging).

**Steps:**
1. Mount Google Drive (if using Colab)
2. Install dependencies
3. Load and prepare training data
4. Fine-tune xlm-roberta-base model
5. Evaluate on test set
6. Save model

## Step 1: Setup & Configuration

In [ ]:
# Set paths
import os
from pathlib import Path

# For Google Colab
COLAB = False  # Set to True if running on Google Colab

if COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set paths for Colab
    DRIVE_TRAIN_FILE = "/content/drive/MyDrive/assets/train_span/data/train.jsonl"
    DRIVE_VAL_FILE = "/content/drive/MyDrive/assets/train_span/data/validation.jsonl"
    DRIVE_TEST_FILE = "/content/drive/MyDrive/assets/train_span/data/test.jsonl"
    MODEL_SAVE_DIR = "/content/drive/MyDrive/assets/train_span/models"
    LOG_DIR = "/content/drive/MyDrive/assets/train_span/logs"
else:
    # Local paths
    DRIVE_TRAIN_FILE = "./data/train.jsonl"
    DRIVE_VAL_FILE = "./data/validation.jsonl"
    DRIVE_TEST_FILE = "./data/test.jsonl"
    MODEL_SAVE_DIR = "./models"
    LOG_DIR = "./logs"

# Create directories if they don't exist
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Training data: {DRIVE_TRAIN_FILE}")
print(f"Validation data: {DRIVE_VAL_FILE}")
print(f"Test data: {DRIVE_TEST_FILE}")
print(f"Model save dir: {MODEL_SAVE_DIR}")
print(f"Colab mode: {COLAB}")

## Step 2: Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'torch',
    'transformers',
    'datasets',
    'pyyaml',
    'numpy',
    'scikit-learn'
]

print("Installing packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

print("\n✓ All dependencies ready!")

## Step 3: Import Libraries & Load Data

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from typing import Dict, List

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict, load_dataset

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 4: Load Training Data

In [ ]:
# Check if files exist
for file_path in [DRIVE_TRAIN_FILE, DRIVE_VAL_FILE, DRIVE_TEST_FILE]:
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            line_count = sum(1 for _ in f)
        print(f"✓ {Path(file_path).name}: {line_count} samples")
    else:
        print(f"✗ {file_path} NOT FOUND")

# Load datasets
print("\nLoading datasets...")
train_dataset = load_dataset('json', data_files=DRIVE_TRAIN_FILE)['train']
val_dataset = load_dataset('json', data_files=DRIVE_VAL_FILE)['train']
test_dataset = load_dataset('json', data_files=DRIVE_TEST_FILE)['train']

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Show sample
print(f"\nSample:")
sample = train_dataset[0]
print(f"  Text: {sample['text'][:100]}...")
print(f"  Sentences: {len(sample['sentences'])}")
print(f"  Has bias: {any(sample['sentence_labels'])}")

## Step 5: Load Model & Tokenizer

In [ ]:
# Configuration
model_name = "xlm-roberta-base"
num_labels = 3  # O, B-BIAS, I-BIAS
id2label = {0: "O", 1: "B-BIAS", 2: "I-BIAS"}
label2id = {"O": 0, "B-BIAS": 1, "I-BIAS": 2}

print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=True
)

print(f"✓ Model loaded successfully")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Device: {next(model.parameters()).device}")

## Step 6: Tokenize & Align Labels

In [ ]:
# Tokenization function
def tokenize_and_align_labels(examples, max_length=512):
    """Tokenize text and align labels with tokens."""
    tokenized_inputs = tokenizer(
        examples['text'],
        truncation=True,
        is_split_into_words=False,
        max_length=max_length,
        padding='max_length',
    )

    labels = []
    for i, label in enumerate(examples['token_labels']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:  # Special tokens
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                if word_idx < len(label):
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(0)
            else:
                if word_idx < len(label):
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(0)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train dataset"
)

val_tokenized = val_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation dataset"
)

test_tokenized = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test dataset"
)

print(f"✓ Tokenization complete")

## Step 7: Setup Training

In [ ]:
# Metrics computation
def compute_metrics(p):
    """Compute evaluation metrics."""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Simple metrics calculation
    tp = sum(1 for pred_seq, true_seq in zip(true_predictions, true_labels)
             for p, t in zip(pred_seq, true_seq) if p == 'B-BIAS' and t == 'B-BIAS')
    fp = sum(1 for pred_seq, true_seq in zip(true_predictions, true_labels)
             for p, t in zip(pred_seq, true_seq) if p == 'B-BIAS' and t != 'B-BIAS')
    fn = sum(1 for pred_seq, true_seq in zip(true_predictions, true_labels)
             for p, t in zip(pred_seq, true_seq) if p != 'B-BIAS' and t == 'B-BIAS')

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': precision, 'recall': recall, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=LOG_DIR,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=3,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    push_to_hub=False,
    seed=42,
    disable_tqdm=False,  # Show progress bar
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

print("✓ Training setup complete")

## Step 8: Train Model

In [ ]:
print("Starting training...")
print(f"Total training samples: {len(train_tokenized)}")
print(f"Total validation samples: {len(val_tokenized)}")
print(f"Batch size: 16")
print(f"Epochs: 3")
print()

train_result = trainer.train()

print("\n✓ Training complete!")
print(f"Training time: {train_result.training_time / 3600:.2f} hours")

## Step 9: Evaluate on Test Set

In [ ]:
print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_tokenized)

print("\nTest Results:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## Step 10: Save Model

In [ ]:
# Save final model
model_save_path = os.path.join(MODEL_SAVE_DIR, 'final_model')
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Save training config
config = {
    'model_name': model_name,
    'num_labels': num_labels,
    'id2label': id2label,
    'label2id': label2id,
    'training_time_hours': train_result.training_time / 3600,
    'test_results': test_results
}

import json
with open(os.path.join(model_save_path, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Model saved to {model_save_path}")
print(f"\nFiles:")
for file in os.listdir(model_save_path):
    file_path = os.path.join(model_save_path, file)
    size = os.path.getsize(file_path) / 1e6  # MB
    print(f"  - {file} ({size:.1f} MB)")

## Step 11: Summary

In [ ]:
print("="*80)
print("TRAINING COMPLETE")
print("="*80)

print(f"\nModel: {model_name}")
print(f"Parameters: {model.num_parameters():,}")
print(f"\nTraining Summary:")
print(f"  Samples: {len(train_tokenized)} train, {len(val_tokenized)} val, {len(test_tokenized)} test")
print(f"  Epochs: 3")
print(f"  Time: {train_result.training_time / 3600:.2f} hours")

print(f"\nTest Results:")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall: {test_results['eval_recall']:.4f}")
print(f"  F1: {test_results['eval_f1']:.4f}")

print(f"\nModel saved to: {model_save_path}")

if COLAB:
    print(f"\nNext step: Use the model for inference!")
    print(f"Model location: {model_save_path}")